# Subset a NISAR RSLC to an area of interest

This notebook cuts a NISAR L1 **RSLC** down to an area of interest (AOI) and saves it as a
smaller RSLC that keeps the full NISAR HDF5 layout, so it can be used as input to
**ISCE3** / the NISAR workflows exactly like a full-size product.

What the subsetter does:

1. Maps the AOI (any WKT/GeoJSON polygon, box or point) into radar coordinates using the
   product's own `metadata/geolocationGrid`, so no DEM or ISCE3 install is needed.
2. Copies every group, dataset and attribute, keeping datatypes (NISAR `complex32`), chunking,
   compression and HDF5 dimension scales. Image layers and per-line datasets are cut to the window.
3. Updates everything that depends on the extent:
   * `zeroDopplerTime` and `slantRange` axes
   * `validSamplesSubSwath*`
   * image statistics
   * scene-centre spacings
   * `zeroDopplerStartTime` / `zeroDopplerEndTime`
   * `boundingPolygon`, `isFullFrame`
   * metadata LUTs (geolocation grid, calibration, noise, Doppler, FM rate), cropped with a margin so they still cover the subset

When the input is a **granule ID**, the file is streamed from ASF and only the HDF5 chunks that
overlap the AOI are read. You don't need to download the full ~30 GB product.

In [ ]:
import contextlib
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import shapely

from nisar_subset import RslcProduct, compute_window, load_aoi, open_source, write_subset

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("nisar_subset").setLevel(logging.INFO)

## 1. Choose the input and the AOI

* **`SOURCE`**: a NISAR RSLC granule ID (searched in NASA CMR) or a path to a local RSLC `.h5`.
* **`AOI`**: WKT, GeoJSON (string, dict or file path), a `(min_lon, min_lat, max_lon, max_lat)`
  box, or `None` to take a small box at the scene centre (handy for a first try).

The offline option uses the small RSLC bundled with this repo for the tests. It is a real
Envisat scene in NISAR RSLC format, with a geolocation grid generated by ISCE3.

In [ ]:
# --- Option A: a NISAR granule (streamed from ASF; needs an Earthdata Login) ---------------
SOURCE = "NISAR_L1_PR_RSLC_004_122_D_067_4005_DHDH_A_20251106T160541_20251106T160622_X05009_N_F_J_001"
AOI = None   # e.g. "POLYGON ((lon lat, lon lat, ...))" or (min_lon, min_lat, max_lon, max_lat)

# --- Option B: offline, the small test RSLC shipped with the repo -------------------------
# SOURCE = "../tests/data/envisat_rslc_fixture.h5"
# AOI = "POLYGON ((-115.62 34.826, -115.58 34.83, -115.59 34.835, -115.61 34.832, -115.62 34.826))"

OUTPUT_DIR = Path("subsets")
POLARIZATIONS = None      # e.g. ["HH"] to keep only HH; None keeps everything
HEIGHTS = None            # terrain heights (m) used to place the AOI; e.g. (0, 3000) in mountains
PAD = 0                   # extra lines/samples around the AOI

OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Log in to Earthdata (granule IDs only)

`earthaccess` reads credentials from `~/.netrc` or the `EARTHDATA_USERNAME` /
`EARTHDATA_PASSWORD` environment variables, or prompts for them. `persist=True` saves them to
`~/.netrc` for next time. You can register at https://urs.earthdata.nasa.gov.

In [ ]:
from nisar_subset.remote import looks_like_granule_id

if looks_like_granule_id(SOURCE):
    import earthaccess
    earthaccess.login(persist=True)

## 3. Open the product and look at the scene

Opening a granule only reads metadata. `RslcProduct` describes what is in the file.

In [ ]:
stack = contextlib.ExitStack()
f, label = stack.enter_context(open_source(SOURCE))
prod = RslcProduct(f)
PRODUCT_PATH, IDENT_PATH = prod.path, prod.identification.name   # kept for after the file is closed

ident = prod.identification
footprint = shapely.from_wkt(ident["boundingPolygon"][()].decode())
print("product      :", prod.path)
print("frequencies  :", prod.frequencies)
print("polarizations:", prod.pols)
print("image shape  :", prod.shape)
print("start / end  :", ident["zeroDopplerStartTime"][()].decode(), "/", ident["zeroDopplerEndTime"][()].decode())
print("footprint    :", footprint.bounds)

In [ ]:
if AOI is None:
    # ~2.5 km box at the centre of the scene
    c = footprint.centroid
    AOI = shapely.box(c.x - 0.015, c.y - 0.012, c.x + 0.015, c.y + 0.012)
aoi = load_aoi(AOI)
print(aoi.wkt)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(*footprint.exterior.xy, label="RSLC footprint")
for g in getattr(aoi, "geoms", [aoi]):
    ax.fill(*g.exterior.xy, alpha=0.5, color="tab:red", label="AOI") if g.geom_type == "Polygon" \
        else ax.plot(*g.xy, "o", color="tab:red", label="AOI")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_aspect("equal", adjustable="datalim")
ax.legend(); ax.set_title("Scene and area of interest");

## 4. Find the radar window

The AOI is mapped into (zero-Doppler time, slant range) with the product's geolocation grid,
at the processor's reference terrain height unless you pass `heights`. The window is the
smallest block of lines and samples that covers the AOI.

In [ ]:
win = compute_window(prod, aoi, heights=HEIGHTS, pad=PAD)
print(win)
for fr in win.cols:
    full = prod.shape[fr]
    frac = win.length * win.width(fr) / (full[0] * full[1])
    print(f"frequency {fr}: {win.length} x {win.width(fr)} samples "
          f"({100 * frac:.2f} % of {full[0]} x {full[1]})")

## 5. Write the subset RSLC

In [ ]:
out = OUTPUT_DIR / (Path(str(label).split("?")[0]).stem + "_subset.h5")
t0 = time.time()
result = write_subset(prod, win, out, polarizations=POLARIZATIONS, overwrite=True, source_label=label)
stack.close()   # done with the input
print(f"wrote {result.output} ({result.output.stat().st_size / 1e6:.1f} MB) in {time.time() - t0:.1f} s")
print(result)

## 6. What changed?

The output has the same groups and datasets as the input. Only datasets that depend on the
image extent have a new shape, and the identification fields describe the new extent.

In [ ]:
def shapes(path):
    out = {}
    with h5py.File(path) as h:
        h.visititems(lambda n, o: out.__setitem__(n, o.shape) if isinstance(o, h5py.Dataset) else None)
    return out

with open_source(SOURCE) as (src, _):
    before = {}
    src.visititems(lambda n, o: before.__setitem__(n, o.shape) if isinstance(o, h5py.Dataset) else None)
after = shapes(result.output)

print(f"datasets in input: {len(before)}, in output: {len(after)}")
print("dropped:", sorted(set(before) - set(after)) or "none")
print("\nresized datasets:")
for k in sorted(after):
    if before.get(k) != after[k]:
        print(f"  {k.split('/', 3)[-1]:75s} {str(before[k]):>16s} -> {after[k]}")

In [ ]:
with h5py.File(result.output) as h:
    p = PRODUCT_PATH  # e.g. /science/LSAR/RSLC
    idn = h[IDENT_PATH]
    for k in ("zeroDopplerStartTime", "zeroDopplerEndTime", "isFullFrame"):
        if k in idn:
            print(f"{k:22s}", idn[k][()].decode())
    print("boundingPolygon       ", idn["boundingPolygon"][()].decode()[:120], "...")
    sw = h[f"{p}/swaths/frequency{result.frequencies[0]}"]
    for k in ("sceneCenterAlongTrackSpacing", "sceneCenterGroundRangeSpacing"):
        print(f"{k:30s} {sw[k][()]:.3f} m")
    pol = result.polarizations[result.frequencies[0]][0]
    print(f"{pol} statistics:", {k: float(v) for k, v in sw[pol].attrs.items() if "value" in k or "dev" in k})

## 7. Quick look

The amplitude of the subset, in radar coordinates, with the subset footprint over the AOI.

In [ ]:
with h5py.File(result.output) as h:
    fr = result.frequencies[0]
    pol = result.polarizations[fr][0]
    slc = h[f"{PRODUCT_PATH}/swaths/frequency{fr}/{pol}"][()]
    amp = np.hypot(slc["r"].astype(np.float32), slc["i"].astype(np.float32)) if slc.dtype.names else np.abs(slc)
    sub_poly = shapely.from_wkt(h[f"{IDENT_PATH}/boundingPolygon"][()].decode())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
db = 20 * np.log10(np.maximum(amp, 1e-6))
a1.imshow(db, cmap="gray", vmin=np.nanpercentile(db, 2), vmax=np.nanpercentile(db, 98), aspect="auto")
a1.set_title(f"{pol} amplitude (dB), radar geometry"); a1.set_xlabel("slant range sample"); a1.set_ylabel("line")
a2.plot(*sub_poly.exterior.xy, label="subset boundingPolygon")
for g in getattr(aoi, "geoms", [aoi]):
    a2.fill(*g.exterior.xy, alpha=0.4, color="tab:red", label="AOI") if g.geom_type == "Polygon" \
        else a2.plot(*g.xy, "o", color="tab:red", label="AOI")
a2.set_aspect("equal", adjustable="datalim"); a2.legend(); a2.set_title("Subset footprint vs AOI")
plt.tight_layout()

## 8. (Optional) Open it with ISCE3

If ISCE3 is installed (`conda install -c conda-forge isce3`), its NISAR reader opens the subset
like any RSLC. `tests/validate_with_isce3.py` in this repo runs a full comparison against the
input product: radar grid, samples, LUT coverage, orbit, and AOI placement with geo2rdr.

In [ ]:
try:
    from nisar.products.readers import SLC
except ImportError:
    print("ISCE3 is not installed in this environment; skipping.")
else:
    rslc = SLC(hdf5file=str(result.output))
    grid = rslc.getRadarGrid()
    print("ISCE3 radar grid:", grid.length, "x", grid.width,
          "| start", grid.sensing_datetime(0), "| near range", round(grid.starting_range, 3), "m")
    print("orbit samples:", rslc.getOrbit().size, "| Doppler LUT:", rslc.getDopplerCentroid().data.shape)

## Same thing from the command line

```bash
subset-rslc NISAR_L1_PR_RSLC_..._001 \
    --aoi "POLYGON ((-147.9 64.8, -147.6 64.8, -147.6 64.9, -147.9 64.9, -147.9 64.8))" \
    -o fairbanks_rslc.h5 --pols HH HV

# or a bounding box, a GeoJSON file, an explicit line/sample window, a local file...
subset-rslc granule.h5 --bbox -147.9 64.8 -147.6 64.9
subset-rslc granule.h5 --aoi my_aoi.geojson --heights 100 1500
subset-rslc granule.h5 --rows 10000 12000 --cols 3000 5000
```
Run `subset-rslc --help` for all options.